In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
merge = pd.read_csv(r"C:\Users\Ifeoluwa\OneDrive\Documents\Finlora\data\processed\finlora_merged_cleaned.csv")

In [4]:
merge['timestamp'] = pd.to_datetime(merge['timestamp'], format='%Y-%m-%d %H:%M:%S')
merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 27 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   transaction_id               126000 non-null  str           
 1   account_id                   126000 non-null  str           
 2   account_type                 126000 non-null  str           
 3   kyc_tier                     126000 non-null  str           
 4   timestamp                    126000 non-null  datetime64[us]
 5   day_of_week                  126000 non-null  str           
 6   hour_of_day                  126000 non-null  int64         
 7   description                  126000 non-null  str           
 8   merchant_name                126000 non-null  str           
 9   merchant_category            126000 non-null  str           
 10  channel                      126000 non-null  str           
 11  amount                       126000 n

In [5]:
merge.groupby('status')['is_fraud'].mean()

status
Completed    0.017412
Declined     0.017587
Reversed     0.332879
Name: is_fraud, dtype: float64

In [6]:
merge['IsWeekend'] = merge['day_of_week'].isin(['Saturday', 'Sunday'])

In [7]:
merge.head()

,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,...,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud,account_holder_name,account_created_date,personal_spend_baseline_usd,IsWeekend
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,...,0,Unknown_id,Unknown_device,903,Completed,0,Amaka Okafor,2022-08-28,63.23,True
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,...,0,Unknown_id,Unknown_device,969,Declined,0,Amaka Okafor,2022-08-28,63.23,False
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,...,0,DEV-9055235108,0.0,970,Completed,0,Amaka Okafor,2022-08-28,63.23,False
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,...,0,Unknown_id,Unknown_device,974,Completed,0,Amaka Okafor,2022-08-28,63.23,False
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,...,0,DEV-9055235108,0.0,1035,Declined,0,Amaka Okafor,2022-08-28,63.23,True


In [8]:
merge = merge.drop(columns=[
    'transaction_id', 'account_id', 'account_holder_name',
    'account_created_date', 'description', 'merchant_name',
    'device_id', 'currency', 'home_country', 'transaction_country',
    'status', 'timestamp', 'day_of_week'
])

In [9]:
merge.head()

,account_type,kyc_tier,hour_of_day,merchant_category,channel,amount,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,is_cross_border,is_new_device,account_age_days,is_fraud,personal_spend_baseline_usd,IsWeekend
0,Individual,Tier3_Enhanced,14,Travel,Card Not Present,84233.00,9.21,9143.33,0,0,Unknown_device,903,0,63.23,True
1,Individual,Tier3_Enhanced,13,Subscription/SaaS,Web Dashboard,9143.33,1.00,9143.33,0,0,Unknown_device,969,0,63.23,False
2,Individual,Tier3_Enhanced,12,Groceries,Mobile App,20639.00,2.26,9143.33,0,0,0.0,970,0,63.23,False
3,Individual,Tier3_Enhanced,14,Subscription/SaaS,Card Not Present,3177.43,0.21,14891.16,0,0,Unknown_device,974,0,63.23,False
4,Individual,Tier3_Enhanced,16,Electronics,Mobile App,97839.52,1.97,49609.59,0,0,0.0,1035,0,63.23,True


In [10]:
merge['IsWeekend'] = merge['IsWeekend'].astype(int)
merge['IsNight'] = merge['hour_of_day'].between(0,5).astype(int)

In [11]:
merge = pd.get_dummies(merge, columns=['kyc_tier','merchant_category', 'account_type', 'channel'], drop_first=True)

In [12]:
merge['HighRiskCombo'] = (
    (merge['transaction_velocity_1h'] > 0) & (merge['amount_to_avg_ratio'] > 5)
).astype(int)

In [13]:
merge.info()
print(merge.shape)

<class 'pandas.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 35 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   hour_of_day                          126000 non-null  int64  
 1   amount                               126000 non-null  float64
 2   amount_to_avg_ratio                  126000 non-null  float64
 3   avg_transaction_amount_30d           126000 non-null  float64
 4   transaction_velocity_1h              126000 non-null  int64  
 5   is_cross_border                      126000 non-null  int64  
 6   is_new_device                        126000 non-null  str    
 7   account_age_days                     126000 non-null  int64  
 8   is_fraud                             126000 non-null  int64  
 9   personal_spend_baseline_usd          126000 non-null  float64
 10  IsWeekend                            126000 non-null  int64  
 11  IsNight                 

In [22]:
merge['is_new_device'].unique()
merge['is_new_device'] = merge['is_new_device'].replace({'0.0': 0, '1.0':1, 'Unknown_device': -1})
merge['is_new_device'] = merge['is_new_device'].astype(int)
merge['is_new_device'].dtype
merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 35 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   hour_of_day                          126000 non-null  int64  
 1   amount                               126000 non-null  float64
 2   amount_to_avg_ratio                  126000 non-null  float64
 3   avg_transaction_amount_30d           126000 non-null  float64
 4   transaction_velocity_1h              126000 non-null  int64  
 5   is_cross_border                      126000 non-null  int64  
 6   is_new_device                        126000 non-null  int64  
 7   account_age_days                     126000 non-null  int64  
 8   is_fraud                             126000 non-null  int64  
 9   personal_spend_baseline_usd          126000 non-null  float64
 10  IsWeekend                            126000 non-null  int64  
 11  IsNight                 

In [23]:
merge = merge.to_csv(r"C:\Users\Ifeoluwa\OneDrive\Documents\Finlora\data\processed\finlora_feature_eng.csv", index=False)